# Track A 03: Llama Hard-Retrieval Difficulty Pilot V3

This is the required **discarded C1-W/C2 pilot**, not the final experiment. It tests whether the harder 4096-token retrieval benchmark is informative before C3-Ref or C4-Ref are inspected.

**Never run the full compression matrix before this pilot decision is reviewed.**

## Why This Second Pilot Exists

Pilots V1 and V2 each achieved only 50% C1-W retrieval. V2 showed that reducing records from 48 to 32 was insufficient: every failure had target first-token rank 3-6 and 10/12 wrong outputs were document distractor codes. V3 keeps 32 records and changes exactly one difficulty factor: **key construction**. It uses eight distinct families with four close sibling keys per family, so every target retains three similarly named distractors without making the entire 32-record namespace nearly identical. Context length, depths, scenario count, scoring, and C1-W/C2 variants remain unchanged. V1 and V2 remain discarded and immutable.

V2 also records first-target-token rank/NLL as a diagnostic; this does not alter the primary retrieval endpoint or the freeze threshold.

## Scientific Protocol

- Model: pinned Llama 3.2-3B AWQ checkpoint.
- Variants: C1-W matched wrapper and C2 custom pyramidal eviction only.
- Pilot scenarios: 12 per variant, balanced across 10%, 50%, and 90% target depths.
- Each document: 32 separated, uniquely coded, similarly named records.
- Prompt length: exactly 4096 input tokens; complete query in the final 32 tokens.
- Primary endpoint: exactly one unique extracted code equal to the target.
- This pilot is discarded and stored separately from all final evidence.

## 1. Install Pinned Dependencies

Run this cell once, then use **Runtime > Restart session**. After restart, continue from the import cell.

In [ ]:
%pip install -q "autoawq==0.2.9" "gptqmodel==7.1.0"
%pip install -q "transformers==5.12.1" accelerate pandas

## 2. Confirm the T4 Runtime

Select **Runtime > Change runtime type > T4 GPU** before continuing.

In [ ]:
!nvidia-smi

## 3. Imports

In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache, set_seed
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.models.llama.modeling_llama import (
    LlamaAttention,
    apply_rotary_pos_emb,
    eager_attention_forward,
    repeat_kv,
)

## 4. Embedded Hard-Retrieval Generator and Scorer

The benchmark implementation is embedded here. The notebook does not import experiment `.py` files.

In [ ]:
"""Deterministic hard-retrieval prompts and scoring for Track A.

The semantic scenarios are model-independent. Materialization uses a supplied
tokenizer so Llama and Qwen can share facts while retaining exact token lengths.
Do not mark a prompt manifest frozen until the C1-W/C2 difficulty pilot passes.
"""

from __future__ import annotations

import hashlib
import json
import random
import re
from pathlib import Path
from typing import Any


GENERATOR_VERSION = "track-a-hard-retrieval-v3-clustered-keys-draft"
MANIFEST_SCHEMA = "cse407-hard-retrieval-manifest-v1"
BASE_SEED = 407_100
DEPTHS = (0.1, 0.5, 0.9)
SCENARIOS_PER_DEPTH = 20
RECORD_COUNT = 32
CONTEXT_TOKENS = 4096
FINAL_QUERY_WINDOW = 32
MIN_RECORD_GAP_TOKENS = 16

CODE_PATTERN = re.compile(
    r"(?<![A-Z0-9])([A-H]\d{3}-[K-NP-Q]\d{2})(?![A-Z0-9])",
    re.IGNORECASE,
)
PARTIAL_CODE_PATTERN = re.compile(
    r"(?<![A-Z0-9])(?:[A-H]\d{1,2}|[A-H]\d{3}(?:-(?:[K-NP-Q]?\d?)?)?)(?![A-Z0-9])",
    re.IGNORECASE,
)
SYSTEM_INSTRUCTION = (
    "Use only the document. Return exactly one activation code and no explanation."
)
PLACEHOLDER = "{HARD_RETRIEVAL_DOCUMENT}"

KEY_FAMILIES = (
    "AURORA", "BOREAL", "CEDAR", "DELTA",
    "EMBER", "FALCON", "GARNET", "HARBOR",
)
KEY_VARIANTS = ("A", "B", "C", "D")
FILLER_TOPICS = (
    "thermal scheduling", "memory allocation", "coastal monitoring",
    "warehouse routing", "battery maintenance", "traffic forecasting",
    "compiler testing", "water treatment", "satellite imaging",
    "clinical logistics", "network telemetry", "acoustic sensing",
    "grid balancing", "crop analysis", "robot calibration", "archive indexing",
)
FILLER_METHODS = (
    "compares recent measurements with a rolling baseline",
    "records each decision before the next review cycle",
    "uses independent checks to detect inconsistent observations",
    "separates routine maintenance from exceptional interventions",
    "prioritizes stable operation over short-lived peak performance",
    "tracks uncertainty alongside every summarized measurement",
    "repeats validation after any material configuration change",
    "keeps raw observations available for later auditing",
)
FILLER_OUTCOMES = (
    "The review found that seasonal variation explained most of the change.",
    "Operators retained the conservative threshold for the next evaluation.",
    "No single measurement was treated as sufficient evidence for deployment.",
    "The team documented both successful trials and inconclusive observations.",
    "A later audit confirmed the ordering but revised the uncertainty estimate.",
    "The report emphasized reproducibility and clearly separated derived values.",
    "Follow-up work focused on failure recovery rather than headline throughput.",
    "The final recommendation required another independent measurement period.",
)


def canonical_hash(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def token_ids_hash(input_ids: list[int]) -> str:
    return hashlib.sha256(",".join(map(str, input_ids)).encode("ascii")).hexdigest()


def _make_unique_codes(rng: random.Random, count: int) -> list[str]:
    letters = "ABCDEFGH"
    suffixes = "KLMNPQ"
    codes: set[str] = set()
    while len(codes) < count:
        codes.add(
            f"{rng.choice(letters)}{rng.randrange(100, 1000):03d}-"
            f"{rng.choice(suffixes)}{rng.randrange(10, 100):02d}"
        )
    result = sorted(codes)
    rng.shuffle(result)
    return result


def _make_similar_keys(rng: random.Random, count: int) -> list[str]:
    batch = rng.randrange(10, 100)
    candidates = [
        f"NODE-{family}-{batch:02d}{variant}"
        for family in KEY_FAMILIES
        for variant in KEY_VARIANTS
    ]
    if count > len(candidates):
        raise ValueError("Clustered-key pilot supports at most 32 records")
    rng.shuffle(candidates)
    return candidates[:count]


def validate_semantic_scenario(scenario: dict[str, Any]) -> None:
    records = scenario["records"]
    if not 32 <= len(records) <= 64:
        raise ValueError("Each scenario must contain 32 to 64 records")
    keys = [record["key"] for record in records]
    codes = [record["code"] for record in records]
    if len(set(keys)) != len(keys) or len(set(codes)) != len(codes):
        raise ValueError("Record keys and codes must be unique")
    if any(CODE_PATTERN.fullmatch(code) is None for code in codes):
        raise ValueError("Every record code must match the declared answer format")
    targets = [record for record in records if record["is_target"]]
    if len(targets) != 1:
        raise ValueError("Each scenario must have exactly one target record")
    target = targets[0]
    if target["key"] != scenario["target_key"] or target["code"] != scenario["target_code"]:
        raise ValueError("Target metadata does not match the target record")
    if scenario["depth"] not in DEPTHS:
        raise ValueError(f"Unsupported target depth: {scenario['depth']}")


def build_semantic_manifest(
    *,
    depths: tuple[float, ...] = DEPTHS,
    scenarios_per_depth: int = SCENARIOS_PER_DEPTH,
    record_count: int = RECORD_COUNT,
    base_seed: int = BASE_SEED,
) -> dict[str, Any]:
    if not 32 <= record_count <= 64:
        raise ValueError("record_count must be between 32 and 64")
    scenarios = []
    for depth_index, depth in enumerate(depths):
        if depth not in DEPTHS:
            raise ValueError(f"Unsupported target depth: {depth}")
        for index in range(scenarios_per_depth):
            scenario_seed = base_seed + depth_index * 100_000 + index
            rng = random.Random(scenario_seed)
            keys = _make_similar_keys(rng, record_count)
            codes = _make_unique_codes(rng, record_count)
            target_index = rng.randrange(record_count)
            records = [
                {
                    "key": key,
                    "code": code,
                    "is_target": position == target_index,
                }
                for position, (key, code) in enumerate(zip(keys, codes))
            ]
            rng.shuffle(records)
            target = next(record for record in records if record["is_target"])
            scenario = {
                "scenario_id": f"d{int(depth * 100):02d}-s{index:02d}",
                "seed": scenario_seed,
                "depth": depth,
                "target_key": target["key"],
                "target_code": target["code"],
                "records": records,
            }
            validate_semantic_scenario(scenario)
            scenarios.append(scenario)

    manifest = {
        "manifest_schema": MANIFEST_SCHEMA,
        "generator_version": GENERATOR_VERSION,
        "status": "DRAFT_UNTIL_C1W_C2_PILOT_PASSES",
        "base_seed": base_seed,
        "depths": list(depths),
        "scenarios_per_depth": scenarios_per_depth,
        "record_count": record_count,
        "scenarios": scenarios,
    }
    manifest["semantic_sha256"] = canonical_hash(manifest)
    return manifest


def _make_filler_text(rng: random.Random, minimum_paragraphs: int = 512) -> str:
    paragraphs = []
    topic_order = list(FILLER_TOPICS)
    method_order = list(FILLER_METHODS)
    outcome_order = list(FILLER_OUTCOMES)
    for index in range(minimum_paragraphs):
        if index % len(topic_order) == 0:
            rng.shuffle(topic_order)
        if index % len(method_order) == 0:
            rng.shuffle(method_order)
        if index % len(outcome_order) == 0:
            rng.shuffle(outcome_order)
        topic = topic_order[index % len(topic_order)]
        method = method_order[index % len(method_order)]
        outcome = outcome_order[index % len(outcome_order)]
        paragraphs.append(
            f"Memo {index + 1} discusses {topic}. The working group {method}. "
            f"Sample batch {2000 + index} was reviewed by {2 + index % 7} analysts. "
            f"{outcome}\n"
        )
    text = "\n".join(paragraphs)
    if CODE_PATTERN.search(text):
        raise AssertionError("Filler text accidentally contains a code-shaped value")
    return text


def _record_text(record: dict[str, Any]) -> str:
    return (
        f"\n[Access record] System {record['key']} uses activation code "
        f"{record['code']}.\n"
    )


def _encode(tokenizer: Any, text: str) -> list[int]:
    return list(tokenizer.encode(text, add_special_tokens=False))


def _allocate_gaps(total: int, count: int, rng: random.Random) -> list[int]:
    minimum = MIN_RECORD_GAP_TOKENS * count
    if total < minimum:
        raise ValueError(
            f"Region has {total} filler tokens but needs at least {minimum} "
            "to keep records separated"
        )
    remaining = total - minimum
    gaps = [MIN_RECORD_GAP_TOKENS] * count
    order = list(range(count))
    rng.shuffle(order)
    for index in range(remaining):
        gaps[order[index % count]] += 1
    return gaps


def _pack_region(
    tokenizer: Any,
    records: list[dict[str, Any]],
    token_budget: int,
    rng: random.Random,
) -> tuple[list[int], str]:
    record_ids = [_encode(tokenizer, _record_text(record)) for record in records]
    filler_budget = token_budget - sum(map(len, record_ids))
    gaps = _allocate_gaps(filler_budget, len(records) + 1, rng)
    filler_text = _make_filler_text(rng)
    filler_ids = _encode(tokenizer, filler_text)
    while len(filler_ids) < filler_budget:
        filler_text += _make_filler_text(rng)
        filler_ids = _encode(tokenizer, filler_text)

    result: list[int] = []
    cursor = 0
    for index, encoded_record in enumerate(record_ids):
        next_cursor = cursor + gaps[index]
        result.extend(filler_ids[cursor:next_cursor])
        result.extend(encoded_record)
        cursor = next_cursor
    result.extend(filler_ids[cursor:cursor + gaps[-1]])
    if len(result) != token_budget:
        raise AssertionError(f"Packed {len(result)} tokens, expected {token_budget}")
    return result, filler_text


def materialize_scenario(
    tokenizer: Any,
    scenario: dict[str, Any],
    *,
    context_tokens: int = CONTEXT_TOKENS,
) -> dict[str, Any]:
    validate_semantic_scenario(scenario)
    question = f"What is the code for {scenario['target_key']}?"
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {
            "role": "user",
            "content": f"<DOC>\n{PLACEHOLDER}\n</DOC>\n{question}",
        },
    ]
    template = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    if template.count(PLACEHOLDER) != 1:
        raise ValueError("Chat template did not preserve the document placeholder exactly once")
    prefix_text, suffix_text = template.split(PLACEHOLDER)
    prefix_ids = _encode(tokenizer, prefix_text)
    suffix_ids = _encode(tokenizer, suffix_text)
    if len(suffix_ids) > FINAL_QUERY_WINDOW:
        raise ValueError(
            f"Question/template suffix is {len(suffix_ids)} tokens; maximum is "
            f"{FINAL_QUERY_WINDOW}"
        )

    document_budget = context_tokens - len(prefix_ids) - len(suffix_ids)
    target_record = next(record for record in scenario["records"] if record["is_target"])
    distractors = [record for record in scenario["records"] if not record["is_target"]]
    target_ids = _encode(tokenizer, _record_text(target_record))
    target_start = round(scenario["depth"] * document_budget)
    after_budget = document_budget - target_start - len(target_ids)
    if target_start <= 0 or after_budget <= 0:
        raise ValueError("Target depth leaves no room on one side of the target record")

    rng = random.Random(scenario["seed"] + 17)
    rng.shuffle(distractors)
    before_count = round(scenario["depth"] * len(distractors))
    before_records = distractors[:before_count]
    after_records = distractors[before_count:]
    before_ids, before_filler = _pack_region(
        tokenizer, before_records, target_start, random.Random(scenario["seed"] + 31),
    )
    after_ids, after_filler = _pack_region(
        tokenizer, after_records, after_budget, random.Random(scenario["seed"] + 47),
    )
    document_ids = before_ids + target_ids + after_ids
    input_ids = prefix_ids + document_ids + suffix_ids
    if len(document_ids) != document_budget or len(input_ids) != context_tokens:
        raise AssertionError("Prompt materialization failed the exact-length gate")

    answer = scenario["target_code"]
    leak_text = SYSTEM_INSTRUCTION + suffix_text + before_filler + after_filler
    if answer.casefold() in leak_text.casefold():
        raise AssertionError("Target answer leaked outside its record")
    document_text = tokenizer.decode(document_ids, skip_special_tokens=False)
    extracted_document_codes = [code.upper() for code in CODE_PATTERN.findall(document_text)]
    expected_codes = {record["code"] for record in scenario["records"]}
    if len(extracted_document_codes) != len(scenario["records"]):
        raise AssertionError("A record was truncated or an extra code appeared in the document")
    if set(extracted_document_codes) != expected_codes:
        raise AssertionError("Materialized document codes differ from semantic records")
    if extracted_document_codes.count(answer) != 1:
        raise AssertionError("Target answer must occur exactly once in the document")

    realized_depth = target_start / document_budget
    if abs(realized_depth - scenario["depth"]) > 1 / document_budget:
        raise AssertionError("Target token depth exceeds one-token rounding tolerance")
    return {
        "scenario_id": scenario["scenario_id"],
        "depth": scenario["depth"],
        "target_key": scenario["target_key"],
        "target_code": answer,
        "record_count": len(scenario["records"]),
        "context_tokens": context_tokens,
        "document_tokens": document_budget,
        "target_start_document_token": target_start,
        "target_start_input_token": len(prefix_ids) + target_start,
        "target_record_tokens": len(target_ids),
        "realized_depth": realized_depth,
        "query_window_tokens": len(suffix_ids),
        "input_ids_sha256": token_ids_hash(input_ids),
        "input_ids": input_ids,
    }


def materialize_prompt_manifest(
    tokenizer: Any,
    semantic_manifest: dict[str, Any],
    *,
    tokenizer_id: str,
    tokenizer_revision: str,
    context_tokens: int = CONTEXT_TOKENS,
    include_input_ids: bool = True,
) -> dict[str, Any]:
    if semantic_manifest.get("semantic_sha256") != canonical_hash(
        {key: value for key, value in semantic_manifest.items() if key != "semantic_sha256"}
    ):
        raise ValueError("Semantic manifest hash is missing or invalid")
    prompts = [
        materialize_scenario(tokenizer, scenario, context_tokens=context_tokens)
        for scenario in semantic_manifest["scenarios"]
    ]
    prompt_set_sha256 = canonical_hash(
        [
            {"scenario_id": prompt["scenario_id"], "sha256": prompt["input_ids_sha256"]}
            for prompt in prompts
        ]
    )
    if not include_input_ids:
        prompts = [
            {key: value for key, value in prompt.items() if key != "input_ids"}
            for prompt in prompts
        ]
    return {
        "manifest_schema": MANIFEST_SCHEMA,
        "generator_version": GENERATOR_VERSION,
        "status": semantic_manifest["status"],
        "semantic_sha256": semantic_manifest["semantic_sha256"],
        "prompt_set_sha256": prompt_set_sha256,
        "tokenizer_id": tokenizer_id,
        "tokenizer_revision": tokenizer_revision,
        "context_tokens": context_tokens,
        "query_window_limit": FINAL_QUERY_WINDOW,
        "prompts": prompts,
    }


def write_manifest(path: str | Path, manifest: dict[str, Any]) -> None:
    destination = Path(path)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    temporary.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    temporary.replace(destination)


def score_response(response: str, target_code: str) -> dict[str, Any]:
    if CODE_PATTERN.fullmatch(target_code) is None:
        raise ValueError(f"Invalid target code: {target_code}")
    extracted = [code.upper() for code in CODE_PATTERN.findall(response)]
    unique_codes = list(dict.fromkeys(extracted))
    target = target_code.upper()
    retrieval_correct = len(unique_codes) == 1 and unique_codes[0] == target
    exact_format = response.strip() == target_code
    legacy_target_anywhere = target.casefold() in response.casefold()

    if retrieval_correct:
        failure_category = "pass"
    elif len(unique_codes) > 1:
        failure_category = "multiple_code"
    elif len(unique_codes) == 1:
        failure_category = "wrong_code"
    elif PARTIAL_CODE_PATTERN.search(response):
        failure_category = "truncated_code"
    else:
        failure_category = "no_code"
    return {
        "retrieval_correct": retrieval_correct,
        "format_correct": exact_format,
        "legacy_target_anywhere": legacy_target_anywhere,
        "failure_category": failure_category,
        "extracted_codes": extracted,
        "unique_extracted_codes": unique_codes,
    }

## 5. CPU-Level Protocol Checks

These checks reject duplicate scenario IDs, malformed scoring, or an incomplete semantic matrix before downloading the model.

In [ ]:
assert score_response("A123-K45", "A123-K45")["retrieval_correct"]
assert not score_response("A123-K45 and B234-L56", "A123-K45")["retrieval_correct"]
assert score_response("A123-K", "A123-K45")["failure_category"] == "truncated_code"
assert score_response("not found", "A123-K45")["failure_category"] == "no_code"

draft = build_semantic_manifest()
assert len(draft["scenarios"]) == 60
assert len({row["scenario_id"] for row in draft["scenarios"]}) == 60
assert all(len(row["records"]) == 32 for row in draft["scenarios"])
print("CPU protocol checks passed; semantic hash:", draft["semantic_sha256"])

## 6. Isolated Paths, Pinned Tokenizer, and Prompt Manifest

All pilot outputs go to `CSE407_llama_track_a_pilot_v3_discarded/`. The cell materializes all prompts with the real pinned tokenizer and fails if any prompt is not exactly 4096 tokens or its query exceeds the final 32-token window.

In [ ]:
SEED = 407
SCHEMA_VERSION = "track-a-llama-hard-pilot-v3-2026-08-03-1"
MODEL_AWQ = "casperhansen/llama-3.2-3b-instruct-awq"
MODEL_AWQ_REVISION = "272b3bde867b606760447deb9a4d2719fbdfd3ae"
CONTEXT_TOKENS = 4096
MAX_NEW_TOKENS = 32
DEFAULT_AVERAGE_BUDGET = 1024
NUM_LAYERS = 28
NUM_Q_HEADS = 24
NUM_KV_HEADS = 8
HEAD_DIM = 128
WINDOW = 32
POOL_KERNEL = 5
PYRAMID_BETA = 20
PILOT_SCENARIO_IDS = [
    f"d{depth:02d}-s{index:02d}"
    for depth in (10, 50, 90)
    for index in (0, 5, 10, 15)
]
PILOT_VARIANTS = ("C1-W", "C2")

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/CSE407_llama_track_a_pilot_v3_discarded")
else:
    SAVE_DIR = Path("/content/CSE407_llama_track_a_pilot_v3_discarded")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = SAVE_DIR / "discarded_pilot_results.csv"
SEMANTIC_MANIFEST_JSON = SAVE_DIR / "draft_semantic_manifest.json"
PROMPT_MANIFEST_JSON = SAVE_DIR / "draft_llama_prompt_manifest.json"
DECISION_JSON = SAVE_DIR / "pilot_decision.json"

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), "A CUDA GPU is required"
assert "T4" in torch.cuda.get_device_properties(0).name, "This pilot is T4-only"
assert transformers.__version__ == "5.12.1"
assert importlib.metadata.version("gptqmodel") == "7.1.0"


def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def append_row(path, row):
    frame = pd.DataFrame([row])
    if path.exists():
        frame = pd.concat([pd.read_csv(path), frame], ignore_index=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def completed_keys(path, fingerprint):
    if not path.exists():
        return set()
    frame = pd.read_csv(path)
    if not (frame["fingerprint"] == fingerprint).all():
        raise RuntimeError("Pilot CSV belongs to a different experiment; move it first")
    return set(map(tuple, frame[["variant", "scenario_id"]].itertuples(index=False, name=None)))


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_AWQ,
    revision=MODEL_AWQ_REVISION,
    trust_remote_code=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

semantic_manifest = build_semantic_manifest()
assert semantic_manifest["generator_version"] == GENERATOR_VERSION
assert len(semantic_manifest["scenarios"]) == 60
write_manifest(SEMANTIC_MANIFEST_JSON, semantic_manifest)
prompt_manifest = materialize_prompt_manifest(
    tokenizer,
    semantic_manifest,
    tokenizer_id=MODEL_AWQ,
    tokenizer_revision=MODEL_AWQ_REVISION,
    context_tokens=CONTEXT_TOKENS,
    include_input_ids=True,
)
write_manifest(PROMPT_MANIFEST_JSON, prompt_manifest)
prompts = {row["scenario_id"]: row for row in prompt_manifest["prompts"]}
assert set(PILOT_SCENARIO_IDS) <= prompts.keys()
assert all(len(prompts[scenario_id]["input_ids"]) == CONTEXT_TOKENS for scenario_id in PILOT_SCENARIO_IDS)

FINGERPRINT_DATA = {
    "schema_version": SCHEMA_VERSION,
    "model_awq": MODEL_AWQ,
    "model_awq_revision": MODEL_AWQ_REVISION,
    "transformers": transformers.__version__,
    "gptqmodel": importlib.metadata.version("gptqmodel"),
    "generator_version": GENERATOR_VERSION,
    "semantic_sha256": semantic_manifest["semantic_sha256"],
    "prompt_set_sha256": prompt_manifest["prompt_set_sha256"],
    "pilot_scenarios": PILOT_SCENARIO_IDS,
    "pilot_variants": PILOT_VARIANTS,
    "context_tokens": CONTEXT_TOKENS,
    "average_budget": DEFAULT_AVERAGE_BUDGET,
}
EXPERIMENT_FINGERPRINT = canonical_hash(FINGERPRINT_DATA)[:16]
(SAVE_DIR / "pilot_environment.json").write_text(
    json.dumps({**FINGERPRINT_DATA, "fingerprint": EXPERIMENT_FINGERPRINT}, indent=2) + "\n"
)
print("Pilot fingerprint:", EXPERIMENT_FINGERPRINT)

## 7. Matched Wrapper and Pyramidal Eviction

C1-W uses this exact wrapper with no eviction. C2 uses the same wrapper with the layerwise eviction schedule, eliminating the stock-versus-wrapper confound during the pilot.

In [ ]:
def load_awq():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_AWQ,
        revision=MODEL_AWQ_REVISION,
        device_map={"": 0},
        attn_implementation="sdpa",
        trust_remote_code=True,
    ).eval()
    config = model.config
    assert config.model_type == "llama"
    assert config.num_hidden_layers == NUM_LAYERS
    assert config.num_attention_heads == NUM_Q_HEADS
    assert config.num_key_value_heads == NUM_KV_HEADS
    assert getattr(config, "head_dim", config.hidden_size // NUM_Q_HEADS) == HEAD_DIM
    raw = config.quantization_config
    quant = raw.to_dict() if hasattr(raw, "to_dict") else dict(raw)
    assert quant["quant_method"] == "awq"
    assert quant["bits"] == 4 and quant["group_size"] == 128
    return model


def build_pyramid_budgets(
    layers=NUM_LAYERS,
    average_budget=DEFAULT_AVERAGE_BUDGET,
    window=WINDOW,
    beta=PYRAMID_BETA,
):
    old_average = int(average_budget) - window
    last = round(old_average / beta)
    first = 2 * old_average - last
    budgets = (np.linspace(first, last, layers).round().astype(int) + window).tolist()
    assert len(budgets) == layers and int(np.mean(budgets)) == average_budget
    assert min(budgets) >= window and all(a >= b for a, b in zip(budgets, budgets[1:]))
    return budgets


PYRAMID_BUDGETS = build_pyramid_budgets()
ACTIVE_EVICTION = False
PKV_LOG = []
_ORIGINAL_LLAMA_FORWARD = LlamaAttention.forward


def select_pyramid_kv(query_states, key_states, value_states, layer_idx, groups, scaling):
    cache_len = key_states.shape[-2]
    budget = min(PYRAMID_BUDGETS[layer_idx], cache_len)
    if cache_len <= budget:
        return key_states, value_states
    obs = min(WINDOW, cache_len)
    old_len = cache_len - obs
    old_keep = max(budget - obs, 0)
    recent_q = query_states[:, :, -obs:, :]
    repeated_k = repeat_kv(key_states, groups) if groups > 1 else key_states
    scores = torch.matmul(recent_q, repeated_k.transpose(2, 3)) * scaling
    causal = torch.triu(
        torch.full((obs, obs), float("-inf"), device=scores.device, dtype=scores.dtype),
        diagonal=1,
    )
    scores[:, :, :, -obs:] += causal[None, None, :, :]
    probabilities = F.softmax(scores, dim=-1, dtype=torch.float32)
    prefix_scores = probabilities[:, :, :, :old_len].sum(dim=-2)
    prefix_scores = prefix_scores.reshape(
        prefix_scores.shape[0], key_states.shape[1], groups, old_len,
    ).mean(dim=2)
    prefix_scores = F.avg_pool1d(
        prefix_scores, kernel_size=POOL_KERNEL, stride=1, padding=POOL_KERNEL // 2,
    )
    topk = prefix_scores.topk(old_keep, dim=-1).indices.sort(dim=-1).values
    gather_idx = topk.unsqueeze(-1).expand(-1, -1, -1, key_states.shape[-1])
    selected_k = torch.cat(
        [key_states[:, :, :old_len, :].gather(2, gather_idx), key_states[:, :, -obs:, :]],
        dim=2,
    ).contiguous()
    selected_v = torch.cat(
        [value_states[:, :, :old_len, :].gather(2, gather_idx), value_states[:, :, -obs:, :]],
        dim=2,
    ).contiguous()
    PKV_LOG.append(
        {"layer": layer_idx, "before": cache_len, "after": selected_k.shape[-2], "budget": budget}
    )
    return selected_k, selected_v


def cache_wrapper_forward(
    self,
    hidden_states,
    position_embeddings=None,
    attention_mask=None,
    past_key_values=None,
    **kwargs,
):
    input_shape = hidden_states.shape[:-1]
    hidden_shape = (*input_shape, -1, self.head_dim)
    query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    cos, sin = position_embeddings
    query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
    is_prefill = past_key_values is not None and past_key_values.get_seq_length(self.layer_idx) == 0
    if past_key_values is not None and not is_prefill:
        key_states, value_states = past_key_values.update(
            key_states, value_states, self.layer_idx,
        )
    attention_interface = ALL_ATTENTION_FUNCTIONS.get_interface(
        self.config._attn_implementation, eager_attention_forward,
    )
    attention_output, attention_weights = attention_interface(
        self,
        query_states,
        key_states,
        value_states,
        attention_mask,
        dropout=0.0,
        scaling=self.scaling,
        **kwargs,
    )
    if is_prefill:
        stored_key, stored_value = key_states, value_states
        if ACTIVE_EVICTION:
            stored_key, stored_value = select_pyramid_kv(
                query_states,
                key_states,
                value_states,
                self.layer_idx,
                self.num_key_value_groups,
                self.scaling,
            )
        past_key_values.update(stored_key, stored_value, self.layer_idx)
    attention_output = attention_output.reshape(*input_shape, -1).contiguous()
    return self.o_proj(attention_output), attention_weights


def enable_wrapper(use_eviction):
    global ACTIVE_EVICTION
    PKV_LOG.clear()
    ACTIVE_EVICTION = bool(use_eviction)
    LlamaAttention.forward = cache_wrapper_forward


def disable_wrapper():
    global ACTIVE_EVICTION
    LlamaAttention.forward = _ORIGINAL_LLAMA_FORWARD
    ACTIVE_EVICTION = False
    PKV_LOG.clear()


def cache_lengths(cache):
    return [int(layer.get_seq_length()) for layer in cache.layers]


def validate_cache(cache, variant, prefill_lengths, decode_tokens):
    expected_prefill = (
        [CONTEXT_TOKENS] * NUM_LAYERS
        if variant == "C1-W"
        else [min(CONTEXT_TOKENS, budget) for budget in PYRAMID_BUDGETS]
    )
    assert prefill_lengths == expected_prefill
    assert cache_lengths(cache) == [length + decode_tokens for length in expected_prefill]
    if variant == "C1-W":
        assert not PKV_LOG
    else:
        assert len(PKV_LOG) == NUM_LAYERS
        assert sorted(row["layer"] for row in PKV_LOG) == list(range(NUM_LAYERS))


def generate_response(model, variant, input_ids):
    use_eviction = variant == "C2"
    enable_wrapper(use_eviction)
    cache = DynamicCache()
    generated = []
    ids = torch.tensor([input_ids], dtype=torch.long, device=model.device)
    try:
        with torch.inference_mode():
            output = model(input_ids=ids, past_key_values=cache, use_cache=True, logits_to_keep=1)
            prefill_lengths = cache_lengths(cache)
            next_token = output.logits[:, -1, :].argmax(-1, keepdim=True)
            for step in range(MAX_NEW_TOKENS):
                generated.append(next_token)
                if next_token.item() in np.atleast_1d(tokenizer.eos_token_id):
                    break
                if step == MAX_NEW_TOKENS - 1:
                    break
                position = torch.tensor([[CONTEXT_TOKENS + step]], device=model.device)
                output = model(
                    input_ids=next_token,
                    position_ids=position,
                    past_key_values=cache,
                    use_cache=True,
                    logits_to_keep=1,
                )
                next_token = output.logits[:, -1, :].argmax(-1, keepdim=True)
        output_ids = torch.cat(generated, dim=1)[0].tolist()
        validate_cache(cache, variant, prefill_lengths, max(0, len(output_ids) - 1))
        return tokenizer.decode(output_ids, skip_special_tokens=True), output_ids
    finally:
        disable_wrapper()
        del cache, ids
        clear_gpu()


def gold_logprob_after_first_token(model, variant, input_ids, answer):
    answer_ids = tokenizer.encode(answer, add_special_tokens=False)
    if tokenizer.decode(answer_ids, skip_special_tokens=True).strip() != answer:
        raise ValueError(f"Answer tokenization is not lossless: {answer}")
    if len(answer_ids) < 2:
        raise ValueError("Answer must contain at least two tokens for the post-first-token metric")
    enable_wrapper(variant == "C2")
    cache = DynamicCache()
    ids = torch.tensor([input_ids], dtype=torch.long, device=model.device)
    log_probabilities = []
    try:
        with torch.inference_mode():
            prefill_output = model(
                input_ids=ids, past_key_values=cache, use_cache=True, logits_to_keep=1,
            )
            prefill_lengths = cache_lengths(cache)
            first_logits = prefill_output.logits[0, -1].float()
            first_target_id = answer_ids[0]
            first_log_probability = float(
                F.log_softmax(first_logits, dim=-1)[first_target_id]
            )
            first_target_rank = int(
                (first_logits > first_logits[first_target_id]).sum().item() + 1
            )
            for index, current_token_id in enumerate(answer_ids[:-1]):
                token = torch.tensor([[current_token_id]], dtype=torch.long, device=model.device)
                position = torch.tensor([[CONTEXT_TOKENS + index]], device=model.device)
                output = model(
                    input_ids=token,
                    position_ids=position,
                    past_key_values=cache,
                    use_cache=True,
                    logits_to_keep=1,
                )
                target_id = answer_ids[index + 1]
                log_probability = F.log_softmax(output.logits[0, -1].float(), dim=-1)[target_id]
                log_probabilities.append(float(log_probability))
        validate_cache(cache, variant, prefill_lengths, len(answer_ids) - 1)
        return {
            "answer_tokens": len(answer_ids),
            "answer_first_token_logprob": first_log_probability,
            "answer_first_token_nll": -first_log_probability,
            "answer_first_token_rank": first_target_rank,
            "answer_first_token_top1": first_target_rank == 1,
            "scored_answer_tokens_after_first": len(log_probabilities),
            "answer_logprob_after_first_sum": sum(log_probabilities),
            "answer_logprob_after_first_mean": float(np.mean(log_probabilities)),
            "answer_nll_after_first_mean": float(-np.mean(log_probabilities)),
        }
    finally:
        disable_wrapper()
        del cache, ids
        clear_gpu()

## 8. Run the Resumable Discarded Pilot V2

The notebook saves each row immediately. It reports first-target-token rank/NLL from prefill and teacher-forced continuation NLL after the first answer token, when the stored cache path is active.

In [ ]:
model = load_awq()
done = completed_keys(RESULTS_CSV, EXPERIMENT_FINGERPRINT)
for scenario_id in PILOT_SCENARIO_IDS:
    prompt = prompts[scenario_id]
    order = np.random.default_rng(SEED + int(scenario_id[1:3]) * 100 + int(scenario_id[-2:])).permutation(
        PILOT_VARIANTS
    )
    for variant in order:
        key = (variant, scenario_id)
        if key in done:
            continue
        response, output_ids = generate_response(model, variant, prompt["input_ids"])
        scoring = score_response(response, prompt["target_code"])
        answer_metric = gold_logprob_after_first_token(
            model, variant, prompt["input_ids"], prompt["target_code"],
        )
        row = {
            "fingerprint": EXPERIMENT_FINGERPRINT,
            "schema_version": SCHEMA_VERSION,
            "variant": variant,
            "scenario_id": scenario_id,
            "depth": prompt["depth"],
            "target_key": prompt["target_key"],
            "target_code": prompt["target_code"],
            "input_ids_sha256": prompt["input_ids_sha256"],
            "response": response,
            "output_tokens": len(output_ids),
            "output_token_ids": json.dumps(output_ids),
            **scoring,
            **answer_metric,
        }
        row["extracted_codes"] = json.dumps(row["extracted_codes"])
        row["unique_extracted_codes"] = json.dumps(row["unique_extracted_codes"])
        append_row(RESULTS_CSV, row)
        print("pilot", key, row["failure_category"], repr(response))

del model
clear_gpu()
results = pd.read_csv(RESULTS_CSV)
assert len(results) == len(PILOT_SCENARIO_IDS) * len(PILOT_VARIANTS)
assert not results.duplicated(["variant", "scenario_id"]).any()
assert results.groupby("scenario_id").input_ids_sha256.nunique().eq(1).all()
scored = results.assign(
    retrieval_correct=results.retrieval_correct.astype(str).str.casefold() == "true"
)
retrieval = scored.groupby("variant").retrieval_correct.mean()
c1w_by_depth = scored[scored.variant == "C1-W"].groupby("depth").retrieval_correct.mean()
c1w_accuracy = float(retrieval["C1-W"])
c2_accuracy = float(retrieval["C2"])
depth_floor_pass = bool((c1w_by_depth >= 0.50).all())
if 0.75 <= c1w_accuracy <= 0.95 and depth_floor_pass:
    decision = "PASS_FREEZE_CANDIDATE"
elif c1w_accuracy == 1.0 and c2_accuracy == 1.0:
    decision = "TOO_EASY_INCREASE_DIFFICULTY"
elif c1w_accuracy < 0.70:
    decision = "TOO_HARD_REDUCE_DIFFICULTY"
else:
    decision = "REVIEW_BEFORE_FREEZE"
decision_record = {
    "fingerprint": EXPERIMENT_FINGERPRINT,
    "status": "DISCARDED_DIFFICULTY_PILOT",
    "decision": decision,
    "c1w_retrieval": c1w_accuracy,
    "c2_retrieval": c2_accuracy,
    "c1w_retrieval_by_depth": {str(key): float(value) for key, value in c1w_by_depth.items()},
    "depth_floor_pass": depth_floor_pass,
    "scenarios_per_variant": len(PILOT_SCENARIO_IDS),
    "freeze_automatic": False,
    "next_action": "Inspect failures, then explicitly freeze or revise the generator.",
    "completed_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
DECISION_JSON.write_text(json.dumps(decision_record, indent=2) + "\n")
print(json.dumps(decision_record, indent=2))

## 9. Decision Gate

V3 may become a freeze candidate only if overall C1-W retrieval is 75%-95% **and every depth is at least 50%** in this small pilot. The depth floor prevents an acceptable aggregate from hiding an impossible target region. All other decisions remain manual; the notebook never freezes prompts automatically.

Do not run C3-Ref or C4-Ref until the accepted prompt manifest is frozen.